# Recovering FEW's mode selection

This example looks at how FEW's `ModeSelector` picks which $(l, m, n)$ harmonic modes to keep in an EMRI waveform, and shows how to recover the same selection by looking the modes up by their $(l, m, n)$ labels.

SEF's stable derivative (`StableDerivative`) uses this mapping: it records the modes selected for the reference waveform, and reuses exactly the same modes for the waveforms at each finite-difference step.

This notebook uses FEW only, not StableEMRIFisher itself.

In [1]:
import numpy as np

import few
from few.utils.modeselector import ModeSelector
from few.trajectory.inspiral import EMRIInspiral
from few.trajectory.ode.flux import KerrEccEqFlux
from few.amplitude.romannet import RomanAmplitude
from few.utils.ylm import GetYlms

use_gpu = False  # set to True to run on an NVIDIA GPU

if not use_gpu:
    # restrict FEW to its CPU backend
    cfg_set = few.get_config_setter(reset=True)
    cfg_set.enable_backends("cpu")

## Trajectory and amplitude modules

The Kerr eccentric equatorial flux trajectory (`KerrEccEqFlux`) and the `RomanAmplitude` amplitude module.

In [2]:
traj = EMRIInspiral(func=KerrEccEqFlux)
amp = RomanAmplitude()

## Select modes with `ModeSelector`

Evolve the trajectory, compute the amplitude of every mode along it and the spin-weighted spherical harmonics $Y_{lm}$ for the viewing angles `(theta, phi)`, then keep only the modes whose contribution to the total power is above `mode_selection_threshold`. A large threshold (0.1) is used so that only a small number of modes survive.

In [3]:
# parameters
m1 = 1e5
m2 = 1e1
a = 0.9
p0 = 10.0
e0 = 0.7
theta = np.pi / 3.0
phi = np.pi / 2.0

# Evolve the trajectory; the last argument is the inclination xI0 = 1.0 (equatorial)
t, p, e, x, Phi_phi, Phi_theta, Phi_r = traj(m1, m2, a, p0, e0, 1.0)

# Amplitudes of every mode along the trajectory, with shape (time steps, modes)
teuk_modes = amp(a, p, e, x)
# Spin-weighted spherical harmonics for each mode, at the viewing angles (theta, phi)
ylm_gen = GetYlms(include_minus_m=False)
ylms = ylm_gen(amp.unique_l, amp.unique_m, theta, phi).copy()[amp.inverse_lm]

mode_selector = ModeSelector(
    amp.l_arr_no_mask, amp.m_arr_no_mask, amp.n_arr_no_mask
)

mode_selection_threshold = 1e-1  # tolerance on mode contribution to total power

modeinds = [amp.l_arr, amp.m_arr, amp.n_arr]
(teuk_modes_in_selector, ylms_in, ls, ms, ns) = mode_selector(
    teuk_modes, ylms, modeinds, mode_selection_threshold=mode_selection_threshold
)

print("selected amplitudes (time steps, modes):", teuk_modes_in_selector.shape)
print("selected Ylms:", ylms_in.shape)

selected amplitudes (time steps, modes): (82, 139)
selected Ylms: (278,)


## Recover the selection by mode label

Build a map from each $(l, m, n)$ label to its column in the full amplitude array, look up the labels returned by `ModeSelector`, and slice out the same columns.

In [4]:
# map each (l, m, n) label to its column in teuk_modes
mode_map = {
    (int(l), int(m), int(n)): idx
    for idx, (l, m, n) in enumerate(zip(amp.l_arr, amp.m_arr, amp.n_arr))
}

# columns of the modes returned by ModeSelector
keep_inds = [mode_map[(int(l), int(m), int(n))] for l, m, n in zip(ls, ms, ns)]

# slice out the same modes
teuk_modes_in_mapping = teuk_modes[:, keep_inds]
ylms_in_mapping = ylms[keep_inds]

## Check

The amplitudes match exactly. The $Y_{lm}$ arrays have different lengths because `ModeSelector` returns two entries per mode (for $+m$ and $-m$), while the mapping keeps only one.

In [5]:
print("amplitudes match:", bool((teuk_modes_in_selector == teuk_modes_in_mapping).all()))
print("Ylms from ModeSelector:", ylms_in.shape, "| from the mapping:", ylms_in_mapping.shape)

amplitudes match: True
Ylms from ModeSelector: (278,) | from the mapping: (139,)


The columns of the kept modes in the full amplitude array:

In [6]:
print(f"{len(keep_inds)} modes kept; first 10 columns: {keep_inds[:10]}")

139 modes kept; first 10 columns: [577, 578, 579, 580, 581, 582, 583, 584, 585, 586]
